# Model 04 — Support Vector Machine (HR Employee Risk Analytics)

Employee attrition, disengagement, and performance decline rarely appear without warning. They accumulate quietly across behavioral signals — attendance patterns, productivity trends, engagement levels, association with quality problems — until the cost becomes visible as a resignation, a disciplinary event, or a performance review. By that point, the intervention opportunity has already passed.

This project applies a **Support Vector Machine (SVM)** to build an early-warning risk score for HR analytics, classifying employees as high-risk or not based on a combination of behavioral, operational, and structural variables. The SVM is the right model here for two reasons: first, the decision boundary between risk and non-risk employees is not linearly separable — it depends on specific combinations of conditions (low engagement *and* high scrap *and* night shift, for instance). The RBF kernel maps these interactions into a higher-dimensional space where separation becomes feasible. Second, the SVM's margin-maximizing objective makes it robust to outliers — essential when working with HR data where individual observations can be atypical without being representative of a broader pattern.

The dataset simulates 1,000 employee records from a manufacturing environment, covering 7 numeric behavioral/operational variables and 3 categorical structural variables. This is not a model for firing decisions. It is a model for prioritizing interventions — identifying which employees need attention before the situation becomes irreversible.

---
**Data source:** `hr_risk_svm_data.csv`  
**Output:** metrics, LinearSVC coefficient analysis, 2D decision boundary maps, and an employee scenario simulator


## Section 1 — Setup

Reproducibility is essential in HR analytics — any model that informs personnel decisions must produce consistent results across runs. We fix a global seed, import all dependencies upfront, and configure visualization defaults once.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. Random state:", RANDOM_STATE)


## Section 2 — Load Data

The dataset contains **1,000 employee records** from manufacturing HR system. Each row represents one employee, described by 10 features across three layers: behavioral indicators (`punctuality_rate`, `engagement_score`), operational performance metrics (`productivity_index`, `scrap_associated_pct`, `training_hours_annual`, `experience_yrs`), and structural context variables (`area_rotation_rate`, `department`, `shift`, `contract_type`). The target `high_risk` is binary — 1 if the employee shows patterns consistent with disengagement or performance decline risk.

> *Note: The dataset used explains a realistic manufacturing HR conditions — where risk is driven by overlapping factors: low engagement combined with high scrap and rotation, outsourcing or temporary contracts under night shift conditions, and limited training in high-turnover departments. The dataset reflects a 30.8% high-risk rate — a meaningful minority that would be missed by any rule-of-thumb threshold approach.*


In [ ]:
try:
    df = pd.read_csv("hr_risk_svm_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/04-SVM-HR-Risk/main/hr_risk_svm_data.csv")
    # FileNotFoundError is intentionally specific — a bare except would silently swallow real data errors

NUM_COLS = ["training_hours_annual", "punctuality_rate", "productivity_index",
            "scrap_associated_pct", "engagement_score", "experience_yrs", "area_rotation_rate"]
CAT_COLS = ["department", "shift", "contract_type"]
TARGET   = "high_risk"

df.head()


## Section 3 — Quick Sanity Checks

Before building any model, we validate the data we actually loaded. We check for missing values, confirm data types, and inspect the class balance. A 30.8% high-risk rate means the dataset is moderately imbalanced — not severe enough to require resampling, but enough that Recall for the risk class is more operationally meaningful than overall Accuracy.


In [ ]:
# Sanity checks. Real datasets usually try to hurt you 🙂
print("Shape:", df.shape)
print("\n--- Data types ---")
df.info()


In [ ]:
print("--- Missing values ---")
print(df.isna().sum())
print("\n--- Target class balance ---")
print(df[TARGET].value_counts())
print("\n--- Risk rate ---")
print(df[TARGET].value_counts(normalize=True).rename({0: "low_risk", 1: "high_risk"}))


In [ ]:
print("--- Categorical variables ---")
for col in CAT_COLS:
    print(f"{col}: {df[col].value_counts().to_dict()}")
print("\n--- Numeric summary ---")
df[NUM_COLS].describe().round(2)


## Section 4 — Exploratory Data Analysis

The EDA here serves a dual purpose: technical validation and operational communication. We want to confirm that the data contains real signal for the model — that high-risk employees look systematically different from low-risk ones across the features. But we also want outputs that an HR manager can read without a statistics background. The bar charts of risk rate by department, shift, and contract type are the most actionable outputs in this section: they identify *where* risk concentrates structurally, independent of individual behavior.


In [ ]:
# Numeric distributions — overlapping histograms by risk class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    axes[i].hist(df[df[TARGET]==0][col], bins=25, alpha=0.6,
                 color="#4C9BE8", edgecolor="white", label="Low Risk")
    axes[i].hist(df[df[TARGET]==1][col], bins=25, alpha=0.6,
                 color="#E8574C", edgecolor="white", label="High Risk")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)
axes[-1].set_visible(False)
plt.suptitle("Numeric Feature Distributions by Risk Class",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Risk rate by categorical variable — the most HR-readable output
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, CAT_COLS):
    rates = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(rates.index, rates.values,
                  color=["#E8574C" if v > df[TARGET].mean() else "#4C9BE8"
                         for v in rates.values], edgecolor="white")
    ax.axhline(df[TARGET].mean(), linestyle="--", color="gray", linewidth=1.2,
               label=f"Overall {df[TARGET].mean():.1%}")
    ax.set_title(f"Risk Rate by {col.replace('_', ' ').title()}")
    ax.set_ylabel("High Risk Rate")
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis="x", rotation=20)
    for bar, v in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f"{v:.1%}", ha="center", fontweight="bold", fontsize=9)
    ax.legend(fontsize=9)
plt.suptitle("Risk Rate by Structural Context Variables",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots by risk class — where do medians and spreads diverge?
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    for val, color in [(0, "#4C9BE8"), (1, "#E8574C")]:
        subset = df[df[TARGET] == val][col]
        axes[i].boxplot([subset.values], positions=[val],
                        patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.7),
                        medianprops=dict(color="white", linewidth=2))
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Low Risk", "High Risk"])
axes[-1].set_visible(False)
plt.suptitle("Feature Distribution by Risk Class (Blue = Low Risk, Red = High Risk)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap on encoded features
df_enc = pd.get_dummies(df, drop_first=True)
corr_target = df_enc.corr()[[TARGET]].sort_values(TARGET, ascending=False)

fig, ax = plt.subplots(figsize=(5, 9))
sns.heatmap(corr_target, annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-0.4, vmax=0.4, linewidths=0.5, ax=ax,
            annot_kws={"size": 9})
ax.set_title("Correlation with High Risk\n(after one-hot encoding)",
             fontweight="bold")
plt.tight_layout()
plt.show()


## Section 5 — Preprocessing

SVM is highly sensitive to feature scale — it computes distances in a high-dimensional feature space, and features with larger absolute values would dominate that distance calculation without standardization. **StandardScaler** normalizes each numeric feature to zero mean and unit variance, ensuring that engagement (range 1–5) and productivity (range 60–130) contribute equally to the decision boundary.

The three categorical variables are **one-hot encoded** with `drop_first=True` to avoid perfect multicollinearity. Everything is wrapped in a **ColumnTransformer** inside a **Pipeline** — the same object handles preprocessing and prediction together, which is essential for hyperparameter tuning (GridSearchCV) without data leakage.


In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# ColumnTransformer: scale numerics, one-hot encode categoricals
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), CAT_COLS)
])

# Stratified split — preserves 30.8% high-risk ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set  : {X_train.shape[0]} employees")
print(f"Test set      : {X_test.shape[0]} employees")
print(f"Risk rate     : train={y_train.mean():.3f}  test={y_test.mean():.3f}")
print(f"Features      : {len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical")


## Section 6 — Train Model (Base SVM + GridSearchCV)

We train in two steps. First, a base SVM with default RBF kernel to establish a performance floor. Then a **GridSearchCV** over kernel type (linear vs RBF), regularization parameter C, and gamma — with 5-fold stratified cross-validation scoring on F1, the metric that balances precision and recall on imbalanced data.

The regularization parameter C controls the margin width: low C creates a wider margin with more tolerance for misclassifications (better generalization); high C forces the model to fit training data more tightly (risk of overfitting). The optimal C is found empirically through grid search.


In [ ]:
# Step 1 — Base SVM (RBF, defaults) for performance floor
svm_base = Pipeline([
    ("pre", preprocessor),
    ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
])
svm_base.fit(X_train, y_train)

print("Base SVM (RBF, default C=1):")
print(f"  Train accuracy: {accuracy_score(y_train, svm_base.predict(X_train)):.4f}")
print(f"  Test  accuracy: {accuracy_score(y_test, svm_base.predict(X_test)):.4f}")
print(f"  Test  F1      : {f1_score(y_test, svm_base.predict(X_test)):.4f}")


In [ ]:
# Step 2 — GridSearchCV: kernel × C × gamma
param_grid = {
    "clf__kernel": ["linear", "rbf"],
    "clf__C":      [0.1, 1, 10, 50],
    "clf__gamma":  ["scale", 0.1, 0.01]  # applies to rbf; sklearn ignores for linear
}

svm_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", SVC(probability=True, random_state=RANDOM_STATE))
])

grid = GridSearchCV(
    svm_pipe,
    param_grid=param_grid,
    scoring="f1",    # F1 is the right scoring target with class imbalance
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best hyperparameters:", grid.best_params_)
print(f"Best CV F1         : {grid.best_score_:.4f}")

best_svm = grid.best_estimator_


## Section 7 — Evaluate

In HR risk analytics, the cost asymmetry of prediction errors is significant:

- **False Negatives** (predicting low-risk for an employee who is actually high-risk): a missed intervention. The employee continues on a deteriorating trajectory — attrition, performance decline, or quality impact — without support.
- **False Positives** (flagging a low-risk employee as high-risk): an unnecessary intervention — an HR conversation that was not strictly needed. Less costly, but still consumes management bandwidth.

This is why we optimize on F1 and pay close attention to Recall for the high-risk class. AUC tells us how well the model *ranks* employees by risk level across all possible thresholds — a threshold-independent view of discriminative power.


In [ ]:
y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Precision (High Risk): {prec:.4f}")
print(f"Recall    (High Risk): {rec:.4f}")
print(f"F1        (High Risk): {f1:.4f}")
print("\n", classification_report(y_test, y_pred, target_names=["Low Risk", "High Risk"]))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ["Low Risk", "High Risk"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.set_title("Confusion Matrix", fontweight="bold")
plt.tight_layout()
plt.show()
print("Rows = what actually happened. Columns = what the model predicted. Diagonal = correct predictions.")


In [ ]:
# ROC + Precision-Recall curves
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color="#4C9BE8", lw=2, label=f"AUC = {auc:.3f}")
axes[0].plot([0,1],[0,1], linestyle="--", color="gray", lw=1, label="Random baseline")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#4C9BE8")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].legend()

axes[1].plot(rec_c, prec_c, color="#E8574C", lw=2, label=f"AP = {ap:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", lw=1,
                label=f"Baseline (risk rate = {y_test.mean():.2f})")
axes[1].fill_between(rec_c, prec_c, alpha=0.1, color="#E8574C")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Predicted probability distribution by true class
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, color="#4C9BE8",
        label="Actual: Low Risk", edgecolor="white")
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, color="#E8574C",
        label="Actual: High Risk", edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", lw=1.5, label="Decision threshold (0.5)")
ax.set_xlabel("Predicted High-Risk Probability")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Risk Class", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## Section 8 — Interpretability: Linear SVM Coefficients

The best SVM uses an RBF kernel — which is powerful but non-interpretable: it does not produce coefficients that map back to the original features. To get directional insight, we train a **companion LinearSVC** on the same data. Linear SVM coefficients have the same directional interpretation as logistic regression coefficients: positive values push toward high risk, negative values are protective.

This is a common pattern in applied ML: use the best-performing model for predictions, and a transparent linear companion to communicate *why* to stakeholders. The two models are trained on the same data with the same preprocessing — the coefficients from the LinearSVC are a reasonable approximation of the directional effects, even when the optimal decision boundary is non-linear.


In [ ]:
# Train LinearSVC as interpretability companion
svm_linear = Pipeline([
    ("pre", preprocessor),
    ("clf", LinearSVC(C=1.0, random_state=RANDOM_STATE, max_iter=5000))
])
svm_linear.fit(X_train, y_train)

# Reconstruct feature names after OHE
ohe = svm_linear.named_steps["pre"].named_transformers_["cat"]
cat_feature_names = list(ohe.get_feature_names_out(CAT_COLS))
all_feature_names = NUM_COLS + cat_feature_names

coef_df = pd.DataFrame({
    "Feature":     all_feature_names,
    "Coefficient": svm_linear.named_steps["clf"].coef_.ravel()
}).sort_values("Coefficient", ascending=False).reset_index(drop=True)

print("LinearSVC Coefficients (positive = pushes toward High Risk):")
print(coef_df.round(4).to_string(index=False))


In [ ]:
# Coefficient bar chart — directional driver visualization
coef_sorted = coef_df.sort_values("Coefficient", ascending=True)
colors = ["#E8574C" if c > 0 else "#4C9BE8" for c in coef_sorted["Coefficient"]]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(coef_sorted["Feature"], coef_sorted["Coefficient"],
               color=colors, edgecolor="white", height=0.65)
ax.axvline(0, color="black", linewidth=1.2)
for bar, val in zip(bars, coef_sorted["Coefficient"]):
    ax.text(val + 0.003 if val >= 0 else val - 0.003,
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=8)
ax.set_xlabel("Coefficient (linear SVM, scaled features)")
ax.set_title("Feature Impact on High-Risk Probability\nRed = Increases Risk  |  Blue = Reduces Risk",
             fontweight="bold")
readable = {f: f.replace("_", " ").title() for f in coef_sorted["Feature"]}
ax.set_yticklabels([readable[f] for f in coef_sorted["Feature"]])
plt.tight_layout()
plt.show()


In [ ]:
# Factor category impact — how much does each dimension contribute?
def categorize_feature(name):
    if name in ["training_hours_annual", "punctuality_rate",
                "productivity_index", "engagement_score", "experience_yrs"]:
        return "Behavioral / Performance"
    if name == "scrap_associated_pct":
        return "Quality"
    return "Structural / Context"

coef_df["Category"] = coef_df["Feature"].apply(categorize_feature)
coef_df["Abs_Coef"] = coef_df["Coefficient"].abs()

cat_impact = (coef_df.groupby("Category")["Abs_Coef"]
              .sum().reset_index()
              .sort_values("Abs_Coef", ascending=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cat_impact["Category"], cat_impact["Abs_Coef"],
              color=["#E8574C", "#4C9BE8", "#F0A500"][:len(cat_impact)],
              edgecolor="white")
for bar, v in zip(bars, cat_impact["Abs_Coef"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)
ax.set_ylabel("Sum of |Coefficients|")
ax.set_title("Risk Driver Impact by Factor Category", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 2D decision boundary maps — SVM trained on two features at a time
# Reveals the non-linear shape of the RBF kernel's decision regions

def plot_2d_boundary(feat_x, feat_y, title, ax):
    X_2d_train = X_train[[feat_x, feat_y]].copy()
    X_2d_test  = X_test[[feat_x, feat_y]].copy()

    svm_2d = Pipeline([
        ("sc", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=5, gamma="scale", probability=True, random_state=RANDOM_STATE))
    ])
    svm_2d.fit(X_2d_train, y_train)

    x_min, x_max = X[feat_x].min() - 0.02, X[feat_x].max() + 0.02
    y_min, y_max = X[feat_y].min() - 0.5,  X[feat_y].max() + 0.5

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = svm_2d.predict(pd.DataFrame({feat_x: xx.ravel(), feat_y: yy.ravel()}))
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, levels=[-0.5, 0.5, 1.5], cmap="bwr")
    for cls, color, label in [(0, "#4C9BE8", "Low Risk"), (1, "#E8574C", "High Risk")]:
        mask = y_test.values == cls
        ax.scatter(X_2d_test.loc[y_test.index[mask], feat_x],
                   X_2d_test.loc[y_test.index[mask], feat_y],
                   color=color, alpha=0.6, s=18, label=label, edgecolors="none")
    ax.set_xlabel(feat_x.replace("_", " ").title())
    ax.set_ylabel(feat_y.replace("_", " ").title())
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_2d_boundary("punctuality_rate",   "scrap_associated_pct", "Punctuality vs Scrap",      axes[0])
plot_2d_boundary("engagement_score",   "scrap_associated_pct", "Engagement vs Scrap",        axes[1])
plot_2d_boundary("punctuality_rate",   "engagement_score",     "Punctuality vs Engagement",  axes[2])
plt.suptitle("SVM RBF Decision Boundaries — 2D Projections",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Summary of Practical Signals

The LinearSVC coefficients reveal a clear, three-layer risk structure:

**Behavioral / Performance drivers (most actionable):**
- **`engagement_score`** is the strongest protective factor — low engagement is the single most predictive behavioral signal for high risk. This is the lever most directly addressable through management and team culture interventions.
- **`punctuality_rate`** carries significant weight — absenteeism and lateness are often leading indicators of disengagement or burnout.
- **`productivity_index`** and **`training_hours_annual`** are both protective — experienced, well-trained employees producing at higher rates are systematically lower risk.

**Quality driver:**
- **`scrap_associated_pct`** is the top positive driver — employees associated with higher scrap rates carry elevated risk. This signal crosses HR and quality engineering: it may indicate skill gaps, attention deficits, or ergonomic/equipment issues that are impacting both performance and engagement.

**Structural / Context drivers:**
- **Night shift** and **outsourcing/temporary contracts** both push risk up. These are structural conditions outside the individual employee's control — they represent systemic risk embedded in how work is organized, not personal deficiencies.
- The **Production department** carries above-average risk; **Quality** and **Maintenance** are protective — consistent with the operational demands and psychological contract differences across these functions.

**Operational implication:** Interventions should be segmented by layer. Structural factors (shift, contract type) require organizational decisions — they cannot be addressed with individual coaching. Behavioral factors (engagement, punctuality) respond to management attention, career development, and workload review. Quality-linked risk (scrap) may need a joint HR + process engineering approach.


## Section 9 — Statistical Validation

With 750 training samples and class imbalance (30.8% high risk), single train/test splits can be noisy. We validate stability with 5-fold cross-validation and a confidence interval for test-set accuracy.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = cross_val_score(best_svm, X_train, y_train, cv=cv, scoring="accuracy")
cv_auc = cross_val_score(best_svm, X_train, y_train, cv=cv, scoring="roc_auc")
cv_f1  = cross_val_score(best_svm, X_train, y_train, cv=cv, scoring="f1")

print("5-Fold Stratified Cross-Validation (training set only):")
print(f"  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  AUC-ROC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")


In [ ]:
# 95% confidence interval for test-set accuracy
n = len(y_test)
p = accuracy_score(y_test, y_pred)
z = 1.96
margin = z * np.sqrt((p * (1 - p)) / n)

print(f"Test-set accuracy : {p:.4f}")
print(f"95% CI            : [{p - margin:.4f}, {p + margin:.4f}]")
print(f"Margin of error   : ±{margin:.4f}")


## Section 10 — Employee Risk Simulator

A simulator turns the model into an HR decision-support tool. Before a quarterly review, a manager can enter the current indicators for a specific employee and receive an estimated risk probability. This enables conditional, evidence-based conversations — not gut-feel assessments.

**Important caveat:** this score is not a verdict. A high probability flags the employee for a *conversation*, not a disciplinary action. The model identifies patterns; context and human judgment determine the response.


In [ ]:
def simulate_employee(
    training_hours_annual: float = 30.0,
    punctuality_rate: float = 0.95,
    productivity_index: float = 100.0,
    scrap_associated_pct: float = 5.0,
    engagement_score: int = 3,
    experience_yrs: int = 5,
    area_rotation_rate: float = 0.10,
    department: str = "Production",
    shift: str = "Morning",
    contract_type: str = "Permanent",
    threshold: float = 0.5
) -> tuple:
    """
    Estimate high-risk probability for an employee profile.

    Parameters
    ----------
    training_hours_annual : Annual training hours (range: 0–80)
    punctuality_rate      : Attendance rate 0–1 (typical: 0.90–1.0)
    productivity_index    : Performance index (range: 60–130, mean ~100)
    scrap_associated_pct  : % scrap linked to employee (range: 0–20)
    engagement_score      : Self-reported engagement 1–5
    experience_yrs        : Years in company (0–30)
    area_rotation_rate    : Department turnover rate (range: 0.0–0.4)
    department            : 'Production', 'Quality', 'Logistics', 'Maintenance', 'Administration'
    shift                 : 'Morning', 'Afternoon', or 'Night'
    contract_type         : 'Permanent', 'Temporary', or 'Outsourcing'
    threshold             : Classification cutoff (default 0.5)
    """
    df_new = pd.DataFrame([{
        "training_hours_annual": training_hours_annual,
        "punctuality_rate":      punctuality_rate,
        "productivity_index":    productivity_index,
        "scrap_associated_pct":  scrap_associated_pct,
        "engagement_score":      engagement_score,
        "experience_yrs":        experience_yrs,
        "area_rotation_rate":    area_rotation_rate,
        "department":            department,
        "shift":                 shift,
        "contract_type":         contract_type
    }])

    prob  = best_svm.predict_proba(df_new)[0, 1]
    clase = int(prob >= threshold)

    print("=" * 58)
    print("  EMPLOYEE RISK PROBABILITY SIMULATOR")
    print("=" * 58)
    print(f"  Department        : {department.upper()}")
    print(f"  Shift             : {shift.upper()}")
    print(f"  Contract type     : {contract_type.upper()}")
    print(f"  Engagement score  : {engagement_score}/5")
    print(f"  Punctuality       : {punctuality_rate:.2f}")
    print(f"  Productivity      : {productivity_index:.0f}")
    print(f"  Scrap %           : {scrap_associated_pct:.1f}%")
    print(f"  Training hrs/yr   : {training_hours_annual:.0f}")
    print(f"  Experience        : {experience_yrs} yrs")
    print(f"  Area rotation     : {area_rotation_rate:.2f}")
    print("-" * 58)
    print(f"  Risk probability  : {prob:.3f} ({prob*100:.1f}%)")
    print(f"  Decision threshold: {threshold:.2f}")
    if clase == 1:
        print("  Prediction: >>> HIGH RISK — SCHEDULE HR CONVERSATION <<<")
    else:
        print("  Prediction: >>> LOW RISK — STANDARD FOLLOW-UP <<<")
    print("=" * 58)
    return prob, clase


### Scenario A — Low-risk: high engagement, experienced, permanent contract


In [ ]:
simulate_employee(
    training_hours_annual=50,
    punctuality_rate=0.99,
    productivity_index=110,
    scrap_associated_pct=2.0,
    engagement_score=5,
    experience_yrs=10,
    area_rotation_rate=0.05,
    department="Quality",
    shift="Morning",
    contract_type="Permanent",
    threshold=0.5
)


### Scenario B — High-risk: disengaged outsourcing, night shift, high scrap


In [ ]:
simulate_employee(
    training_hours_annual=5,
    punctuality_rate=0.78,
    productivity_index=80,
    scrap_associated_pct=10.0,
    engagement_score=1,
    experience_yrs=1,
    area_rotation_rate=0.30,
    department="Production",
    shift="Night",
    contract_type="Outsourcing",
    threshold=0.5
)


### Scenario C — What-if: same high-risk profile, but engagement improved to 4


In [ ]:
# Isolates the marginal effect of a targeted engagement intervention
simulate_employee(
    training_hours_annual=5,
    punctuality_rate=0.78,
    productivity_index=80,
    scrap_associated_pct=10.0,
    engagement_score=4,         # only change — engagement intervention
    experience_yrs=1,
    area_rotation_rate=0.30,
    department="Production",
    shift="Night",
    contract_type="Outsourcing",
    threshold=0.5
)


## Final Reflection

This project is not about predicting who will leave or fail.  
It is about **making disengagement and performance decline visible before they become irreversible**.

HR risk does not follow a linear rule. An employee can have poor punctuality without being high risk if their engagement and productivity are strong. An outsourcing worker on night shift can be excellent. Risk emerges from specific *combinations* of conditions — which is exactly why a kernel-based model like SVM captures it better than a simple threshold on any single variable.

What this model offers is not a verdict, but a **conversation**:
- *What happens to risk if we move this employee to morning shift?*
- *How much does engagement improvement reduce probability?*
- *Which departments are structurally generating risk, independent of individual behavior?*

These questions matter more than the AUC score.

---

*LozanoLsa  |  Operational Excellence · MBB · Machine Learning  |  GitHub: LozanoLsa*
